# Music Studio Dataset — automatic private save

**Accelerator None / CPU build / Internet ON.** One cell checks your Kaggle
credentials, compiles the pinned x86-64/T4 runtime, verifies every model and
submits **one complete private Dataset** through the same native Kaggle
streaming uploader the other JJP notebooks use. No model passes through your PC.

1. Enable your private Kaggle Secret `KAGGLE_API_TOKEN` (or `KAGGLE_USERNAME`
   and `KAGGLE_KEY`). Never paste a token into code or chat.
2. Leave `DATASET_OWNER` blank when authentication reveals your username;
   otherwise set only your Kaggle username.
3. Run the single code cell. It prints **PRIVATE MUSIC STUDIO DATASET SUBMITTED**
   with a link. The first compilation is the long step; the cell stays busy.
4. Wait for Kaggle to process the Dataset, then attach that ONE Dataset to
   **Music-Studio.ipynb** through **Add Input → Your Datasets**.

The saved set is **18 pinned files / 24.77 GB** plus the compiled
runtime: caption, companion, cover, generation, karaoke, midi, stems, tempo, whisper, writer. Each file has an immutable source revision,
exact byte count and SHA-256. The compiled runtime is built once here on CPU and
reused by every later GPU session, which never compiles it.

Space is measured, not assumed. The build picks the first scratch location with
enough free room, and upload needs only the largest missing file
(**caption-moss-lm-q8_0.gguf**, 8.71 GB) plus a reserve, so the
Dataset is never held twice on local disk. An attached, checksum-verified Music
Studio runtime is reused instead of recompiling. Only this builder's own
temporary downloads are released; attached inputs are never touched.

One private create request per run, never an update or overwrite of an existing
Dataset. An interrupted final request has an uncertain outcome: check the
Dataset page before retrying. Receipts and the build log stay under
`/kaggle/working/music_studio_dataset` and the chosen scratch location.

Embedded builder sources: `70769346939e2870ae29c1b4b57e92d2f614563ad33eb80b2e1c0fabf00cd4ef`
· [Setup and validation](https://github.com/bothashaun-maker/Music/blob/main/docs/music-studio/README.md)
· [Complete tool guide](https://github.com/bothashaun-maker/Music/blob/main/Music-Studio-Guide.pdf)


In [ ]:
DATASET_OWNER = ''  # Optional: your Kaggle username, never an API key.
BUILD_JOBS = 2      # CPU compilation; lower to 1 if the build reports insufficient memory.

from pathlib import Path
import base64, hashlib, importlib, io, json, sys, zipfile

SUPPORT = Path('/kaggle/working/music-studio-dataset-support') / '70769346939e'
payload = base64.b64decode('...')
assert hashlib.sha256(payload).hexdigest() == '70769346939e2870ae29c1b4b57e92d2f614563ad33eb80b2e1c0fabf00cd4ef', 'Embedded Music Studio source checksum failed'
with zipfile.ZipFile(io.BytesIO(payload)) as archive:
    for name in archive.namelist():
        relative = Path(name)
        if relative.is_absolute() or '..' in relative.parts or relative.parts[0] != 'music_studio':
            raise RuntimeError('Invalid embedded source path: ' + name)
        target = SUPPORT / relative
        target.parent.mkdir(parents=True, exist_ok=True)
        target.write_bytes(archive.read(name))
sys.path.insert(0, str(SUPPORT))
importlib.invalidate_caches()
for name in [n for n in sys.modules if n == 'music_studio' or n.startswith('music_studio.')]:
    sys.modules.pop(name, None)  # Rerunning this cell must not reuse stale sources.

from music_studio.common import GIB, choose_scratch
from music_studio.dataset import SCRATCH_CANDIDATES, authenticated_api, create_dataset, username

catalog = json.loads((SUPPORT / 'music_studio/catalog.json').read_text())
api = authenticated_api()  # Check credentials before any compile, download or request.
owner = DATASET_OWNER or username(api)
if not owner:
    raise RuntimeError('Set DATASET_OWNER to your Kaggle username, then rerun. Nothing was built or downloaded.')
scratch = choose_scratch(8 * GIB, SCRATCH_CANDIDATES, 'music-studio-build')
cache = choose_scratch(max(a['bytes'] for a in catalog['assets']) + GIB, SCRATCH_CANDIDATES, 'music-studio-downloads')
result = create_dataset(catalog, '/kaggle/working/music_studio_dataset', Path('/kaggle/input'),
                        cache, owner, jobs=BUILD_JOBS, api=api, build_scratch=scratch)
print('Dataset handle:', result['handle'])
print('Local receipt:', result['receipt'])
